# geo0442 – Scraping the DGM1 Tile Index of NRW

**Unit:** geo0440 GeoPandas: DTM tiles and bounding boxes · **Builds on:** geo0441 (tile names → tile polygons, `tiles_covering`) · **Data:** none to prepare; the file list (about 3.5 MB) and a few tiles (about 2.5 MB each) come from opengeodata.nrw.de

In geo0441 you computed tile *names* from coordinates, but not the survey year at the end of the name. Only the server knows it. Here you read the list of all files the server offers, turn each of the 36 000 names into a 1 km square, and get an **index map** of the NRW terrain model: where the tiles are, when they were surveyed, how large they are. Then you select the tiles for an area with a spatial query and download them.

You will learn to
1. download a web page with **requests** and check what came back (status, type, encoding, size), and keep a local copy,
2. read **XML** with **BeautifulSoup**: tags and attributes,
3. parse file names with a **regular expression** and check that every name fits,
4. build 36 000 polygons in one step (vectorised) and save a **tile index** GeoPackage,
5. select tiles with a **spatial query** and download them politely (streamed, skip existing files, check the size).

## 0. Setup

In [ ]:
import json
import os
import re
import time
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import requests
import shapely
from shapely import affinity
from shapely.geometry import box
from bs4 import BeautifulSoup

def course_data(start=None):
    """The data/ folder of your course folder (outside the session folders, ignored by git).
    OPENGEO_DATA if set; else the folder above the session folder sNN_date; else ./data."""
    env = os.environ.get("OPENGEO_DATA")
    if env:
        return Path(env).expanduser()
    start = Path.cwd() if start is None else Path(start)
    for folder in (start, *start.parents):
        if re.fullmatch(r"s\d{2}_\d{4}-\d{2}-\d{2}", folder.name):
            return folder.parent / "data"
    return Path("data")


CRS = "EPSG:25832"                  # ETRS89 / UTM zone 32N, the CRS of all NRW geodata
TILE = 1000                         # NRW DTM tiles are 1 km x 1 km
DGM1_URL = "https://www.opengeodata.nrw.de/produkte/geobasis/hm/dgm1_tiff/dgm1_tiff/"
HEADERS = {"User-Agent": "opengeo teaching notebook geo0442 (python-requests)"}   # say who you are

DATA = course_data()                                 # e.g. ~/iecs9544-ws2026/data (hub) or <course folder>/data (laptop)
ORIGINAL = DATA / "original" / "dgm1_tiff"           # downloads as they come from the server
OUT = DATA / "derived" / "geo0442"                   # results of this notebook (zip for the laptop)
INDEX_XML = ORIGINAL / "index.xml"
INDEX_GPKG = DATA / "derived" / "dgm1_tile_index.gpkg"   # the tile index, shared with the QGIS script geo0443

for folder in (ORIGINAL, OUT, INDEX_GPKG.parent):
    folder.mkdir(parents=True, exist_ok=True)
print("data folder:", DATA.resolve())
print("requests", requests.__version__, "| geopandas", gpd.__version__, "| shapely", shapely.__version__)

## 1. What does the server send?
Open [the DGM1 download page](https://www.opengeodata.nrw.de/produkte/geobasis/hm/dgm1_tiff/dgm1_tiff/) in your browser. You see a table with file names. Now show the page source (*Ctrl+U*): the browser has formatted a list that is really **XML**.

A script should download such a list **once** and keep a copy: the server is shared by everybody, and you will run the notebook more than once. `fetch_index` reads the local copy if there is one and downloads only if there is none (or if you ask for it with `refresh=True`).

**Predict:** the list has about 36 000 lines like `<file name="dgm1_32_280_5652_1_nw_2022.tif" size="2393341" timestamp="…" />`. How many MB is that?

In [ ]:
def fetch_index(path=INDEX_XML, refresh=False):
    """The DGM1 file list as text: the local copy, or downloaded once and saved (with its HTTP headers).
    refresh=True asks the server whether the list has changed since our copy (conditional GET)."""
    info_file = path.with_suffix(".json")
    saved = json.loads(info_file.read_text()) if info_file.exists() else {}
    if path.exists() and not refresh:
        print(f"local copy {path} ({path.stat().st_size / 1e6:.1f} MB), downloaded {saved.get('downloaded', '?')}")
        return path.read_bytes().decode("utf-8")
    headers = dict(HEADERS)
    if path.exists():                             # "send the list only if it differs from my copy"
        if saved.get("etag"):
            headers["If-None-Match"] = saved["etag"]
        if saved.get("last_modified"):
            headers["If-Modified-Since"] = saved["last_modified"]
    r = requests.get(DGM1_URL, headers=headers, timeout=60)
    now = f"{datetime.now():%Y-%m-%d %H:%M}"
    if r.status_code == 304:                      # Not Modified: no body, our copy is current
        saved["checked"] = now
        info_file.write_text(json.dumps(saved, indent=1))
        print(f"304 Not Modified: the local copy (list of {saved.get('last_modified')}) is current")
        return path.read_bytes().decode("utf-8")
    r.raise_for_status()                          # stop here on 404, 500, ...
    path.write_bytes(r.content)                   # bytes exactly as the server sent them
    info_file.write_text(json.dumps({"url": DGM1_URL, "downloaded": now, "checked": now,
                                     "last_modified": r.headers.get("Last-Modified"),
                                     "etag": r.headers.get("ETag")}, indent=1))
    print(f"downloaded {len(r.content) / 1e6:.1f} MB in {r.elapsed.total_seconds():.1f} s")
    return r.content.decode("utf-8")


xml = fetch_index()
print(xml[:600])


Look at the response itself once (a second, small request: only the headers, `HEAD` instead of `GET`):

In [ ]:
r = requests.head(DGM1_URL, headers=HEADERS, timeout=30)
for h in ("Content-Type", "Content-Length", "Last-Modified", "ETag"):
    print(f"{h:15s} {r.headers.get(h)}")
print("status", r.status_code, "| requests would decode the text as", r.encoding)

Four things to note:
* **Status 200** means OK. `raise_for_status()` turns 404 (not found) or 500 (server error) into an error, so a script never continues with an error page.
* **`text/xml`**: XML uses the same angle brackets as HTML, but its own tag names. The tree here: `opengeodata` → `datasets` → `dataset` → `files` → many `file` tags.
* **Encoding.** The server does not say which character set it uses, so requests assumes ISO-8859-1 for `r.text`. The first line of the file says `encoding="UTF-8"`. Compare `r.text` and `r.content.decode("utf-8")` on the word *Geländemodell* in the cell below. That is why `fetch_index` decodes the bytes itself.
* **`Last-Modified`** is the date of the list. Your index is a snapshot of that day: write the date into your results (§4.3).
* **`ETag`** is a fingerprint of this version of the list. `fetch_index` saves both in `index.json`; with `refresh=True` it sends them back (`If-None-Match`, `If-Modified-Since`). If nothing has changed, the server answers **304 Not Modified** without a body: checking costs a few hundred bytes instead of 3.5 MB.

In [ ]:
r = requests.get(DGM1_URL, headers=HEADERS, timeout=60, stream=True)   # stream: read only the first bytes
first = next(r.iter_content(chunk_size=400))
r.close()
print("guessed ISO-8859-1:", first.decode(r.encoding)[-150:-60])
print("declared UTF-8:    ", first.decode("utf-8", errors="replace")[-150:-60])

In [ ]:
xml = fetch_index(refresh=True)      # ask the server: changed since my copy? (run it twice)


## 2. Read the XML with BeautifulSoup
BeautifulSoup turns the text into a tree of **tags**. A tag has a name (`file`), **attributes** (`name="…" size="…"`), which you read like a dictionary, and possibly children and text.

* `soup.find("dataset")`: the first tag with that name (or `None`),
* `soup.find_all("file")`: a list of all of them,
* `tag["name"]` or `tag.attrs`: its attributes, always as **text**.

**Questions** (read the [BeautifulSoup documentation](https://www.crummy.com/software/BeautifulSoup/bs4/doc/) for 5 minutes): What is a tag? Which attributes does the tag `file` have? What do `find` and `find_all` return if nothing matches?

In [ ]:
try:
    soup = BeautifulSoup(xml, features="xml")          # XML parser (needs the package lxml)
except Exception as e:                                 # bs4.FeatureNotFound without lxml
    print("no lxml, using html.parser:", e)
    soup = BeautifulSoup(xml, features="html.parser")  # also finds the <file> tags here

dataset = soup.find("dataset")
print(dataset["name"], "|", dataset["title"])
print("metafile:", soup.find("metafile").attrs)

files = soup.find_all("file")
print(len(files), "<file> tags; the first one:")
print(files[0])
print(files[0].attrs)

### 2.1 All file tags as a table
One row per `file` tag. Attribute values are text: convert `size` to an integer (bytes) and `timestamp` to a date yourself.

**Exercise:** write `files_table(soup)`. It returns a DataFrame with one row per `file` tag and the columns `file` (the name, text), `size_bytes` (integer) and `timestamp` (`pd.to_datetime`). Build a list of dictionaries first, one per tag.

In [ ]:
def files_table(soup):
    """DataFrame of all <file> tags: file (text), size_bytes (int), timestamp (datetime)."""
    # YOUR CODE HERE


files_df = files_table(soup)
assert list(files_df.columns) == ["file", "size_bytes", "timestamp"]
assert files_df["size_bytes"].dtype.kind == "i" and files_df["timestamp"].dtype.kind == "M"
files_df.head()

In [ ]:
print(f"{len(files_df)} files, {files_df.size_bytes.sum() / 1e9:.1f} GB, "
      f"{files_df.size_bytes.min() / 1e6:.1f}–{files_df.size_bytes.max() / 1e6:.1f} MB per file")
print("server timestamps from", files_df.timestamp.min().date(), "to", files_df.timestamp.max().date())

The `timestamp` is the date the **file** was written on the server, not the date of the survey flight. The survey year is in the name. That is the next step.

## 3. Parse the file names
In geo0441 you split the name at `_` and took parts 2 and 3. That works, but it also "works" on a wrong name: `split` never complains. A **regular expression** describes the whole name, so a name that does not fit is noticed instead of being read wrongly:

```
^dgm1_32_(?P<e_km>\d{3})_(?P<n_km>\d{4})_1_nw_(?P<year>\d{4})\.tif$
```
`^…$` the whole name · `\d{3}` three digits · `(?P<e_km>…)` a *named group* · `\.` a real dot (a bare `.` means "any character").

In [ ]:
NAME = re.compile(r"^dgm1_32_(?P<e_km>\d{3})_(?P<n_km>\d{4})_1_nw_(?P<year>\d{4})\.tif$")

print(NAME.match("dgm1_32_324_5724_1_nw_2025.tif").groupdict())
print(NAME.match("dgm1_32_324_5724_1_nw.xyz.gz"))          # an old XYZ name: no match -> None

**Exercise:** write `tile_key(e_km, n_km)` (as in geo0441) and `parse_name(name)`. `parse_name` returns a dictionary with `key` (text, the name without the year), `e_km`, `n_km` and `year` (integers), and raises a `ValueError` with the name in the message if `NAME` does not match. Why is it better to raise an error than to return `None`?

In [ ]:
def tile_key(e_km, n_km):
    """Tile name without the survey year (as in geo0441); a text identifier."""
    # YOUR CODE HERE


def parse_name(name):
    """'dgm1_32_324_5724_1_nw_2025.tif' -> {'key': 'dgm1_32_324_5724_1_nw', 'e_km': 324, 'n_km': 5724, 'year': 2025}.
    Raises ValueError for a name that does not follow the pattern."""
    # YOUR CODE HERE


assert parse_name("dgm1_32_324_5724_1_nw_2025.tif") == {"key": "dgm1_32_324_5724_1_nw", "e_km": 324, "n_km": 5724, "year": 2025}
try:
    parse_name("dgm1_32_324_5724_1_nw.xyz.gz")
    print("no error: check your function")
except ValueError as e:
    print("ValueError:", e)

Now all names. Check first, then parse: a scraper must tell you when the server has changed something.

**Predict:** do all 36 000 names fit the pattern? Can a key (a place) appear twice?

In [ ]:
fits = files_df["file"].str.match(NAME)
print(f"{fits.sum()} names fit, {(~fits).sum()} do not")
if (~fits).any():
    print(files_df.loc[~fits, "file"].head(10).tolist())     # look at them before you drop them

idx = pd.concat([files_df[fits].reset_index(drop=True),
                 pd.DataFrame([parse_name(f) for f in files_df.loc[fits, "file"]])], axis=1)
assert idx["key"].is_unique, "a tile appears twice"
idx.head()

## 4. From 36 000 names to an index map
### 4.1 One polygon per tile: loop or vectorised?
`tile_box` from geo0441 builds one square per call. shapely 2 can also build all squares in **one** call from arrays of coordinates (`shapely.box`): the loop runs in C instead of Python.

**Predict:** how long does the Python loop take for 36 000 squares, and how much faster is the vectorised version?

In [ ]:
def tile_box(e_km, n_km, size_km=1):
    """Polygon of the DTM tile with the lower left corner (e_km, n_km) in kilometres, EPSG:25832 (geo0441)."""
    return box(e_km * 1000, n_km * 1000, (e_km + size_km) * 1000, (n_km + size_km) * 1000)


t0 = time.perf_counter()
geoms_loop = [tile_box(e, n) for e, n in zip(idx["e_km"], idx["n_km"])]
t1 = time.perf_counter()
x0, y0 = idx["e_km"].to_numpy() * TILE, idx["n_km"].to_numpy() * TILE
geoms = shapely.box(x0, y0, x0 + TILE, y0 + TILE)                   # all squares at once
t2 = time.perf_counter()

assert shapely.equals(geoms_loop, geoms).all()
print(f"loop {t1 - t0:.3f} s, vectorised {t2 - t1:.4f} s ({(t1 - t0) / (t2 - t1):.0f} times faster)")

### 4.2 The tile index as a GeoDataFrame
The numbers are already metres in UTM 32N: **assign** the CRS (geo0221 §2).

**Predict:** NRW has an area of 34 112 km². More or fewer than 34 112 tiles?

In [ ]:
tiles = gpd.GeoDataFrame(idx, geometry=geoms, crs=CRS)
tiles["size_mb"] = (tiles["size_bytes"] / 1e6).round(2)
tiles = tiles[["key", "file", "year", "e_km", "n_km", "size_mb", "size_bytes", "timestamp", "geometry"]]

print(f"{len(tiles)} tiles = {tiles.area.sum() / 1e6:.0f} km², {tiles.size_mb.sum() / 1000:.1f} GB")
print("extent (m):", tiles.total_bounds)
tiles.groupby("year").agg(tiles=("key", "size"), size_gb=("size_mb", lambda s: round(s.sum() / 1000, 1)))

More tiles than km²: along the border a tile also covers parts of the neighbours (Netherlands, Belgium, Lower Saxony, Hesse, Rhineland-Palatinate).

Now the map. 36 000 polygons are too many for a web map (`explore()` would send all of them to your browser), a static plot is fast.

**Look:** do the survey years form blocks? Where was NRW surveyed most recently? Where is the camp of geo0441 (red dot)?

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 9))
tiles.plot(ax=ax, column="year", categorical=True, cmap="viridis", edgecolor="none",
           legend=True, legend_kwds={"title": "survey year", "loc": "lower left", "fontsize": 12, "title_fontsize": 12})
ax.plot(324_880, 5_724_300, "o", color="red", markersize=8)          # Vetera I (geo0441 CENTRE)
ax.set_title(f"DGM1 tiles of NRW ({len(tiles)} tiles of 1 km², opengeodata.nrw.de)", fontsize=14)
ax.set_xlabel("E (m), EPSG:25832", fontsize=12)
ax.set_ylabel("N (m)", fontsize=12)
ax.ticklabel_format(style="plain")
plt.show()

### 4.3 Save the index for QGIS
One GeoPackage, two layers: the tiles, and a small table that says where and when the index was made (without it, nobody knows in a year how old your index is).

In [ ]:
info_file = INDEX_XML.with_suffix(".json")
saved = json.loads(info_file.read_text()) if info_file.exists() else {}
info = pd.DataFrame([{"source": DGM1_URL, "last_modified": saved.get("last_modified"),
                      "etag": saved.get("etag"), "downloaded": saved.get("downloaded"), "tiles": len(tiles),
                      "licence": "Geobasis NRW, dl-de/zero-2-0"}])

gpkg = INDEX_GPKG
tiles.to_file(gpkg, layer="dgm1_tiles", driver="GPKG")

import pyogrio
pyogrio.write_dataframe(info, gpkg, layer="index_info")               # a table without geometry
print(pyogrio.list_layers(gpkg), f"{gpkg.stat().st_size / 1e6:.1f} MB")
info

In QGIS: *Layer → Add Layer → Add Vector Layer…* → `data/derived/dgm1_tile_index.gpkg` (in the data folder printed in §0), layer `dgm1_tiles`. *Symbology → Categorized* by `year`, label with `key`. Compare with `dtm_tiles_camp_500m` from geo0441: the squares lie on top of each other. This index is what QGIS tile-download plugins build internally. The QGIS script geo0443 builds and uses the same file, so on the laptop you can select tiles with the mouse.

## 5. Select tiles with a spatial query
In geo0441 `tiles_covering` *computed* the squares around an area. With the index we *look them up*, and get the real file name with the year. A **spatial index** (`tiles.sindex`, an R-tree) finds candidates by bounding box first and tests the exact geometry only for those.

The area: the camp of geo0441 plus 500 m. If you have run geo0441, its GeoPackage is read; otherwise the model is built again from the same numbers.

In [ ]:
GEO0441 = DATA / "derived" / "geo0441_vetera.gpkg"

if GEO0441.exists():
    camp = gpd.read_file(GEO0441, layer="camp_model").to_crs(CRS).union_all()
    print("camp from", GEO0441)
else:                                               # the geo0441 model: centre, 945 m x 625 m, 5 deg, r = 30 m
    e, n, length, width, r = 324_880, 5_724_300, 945, 625, 30
    card = box(e - width / 2 + r, n - length / 2 + r, e + width / 2 - r, n + length / 2 - r).buffer(r)
    camp = affinity.rotate(card, 5, origin=(e, n))
    print("camp model rebuilt (geo0441 numbers)")
area = camp.buffer(500)
print(f"area {area.area / 1e6:.2f} km²")

**Predict:** geo0441 found 6 tiles for the camp plus 500 m. Same here? How many MB will the download be?

**Exercise:** write `select_tiles(index, geom)`. It returns the rows of the tile index whose square **intersects** `geom`, sorted by `key`. Use the spatial index: `index.sindex.query(geom, predicate="intersects")` returns the row *positions* (→ `iloc`). Then compare with a query without spatial index (`index[index.intersects(geom)]`): same result? Which one is faster (`%timeit`)?

In [ ]:
def select_tiles(index, geom):
    """Rows of the tile index whose square intersects `geom` (same CRS), sorted by key."""
    # YOUR CODE HERE


sel = select_tiles(tiles, area)
assert sel["key"].tolist() == tiles[tiles.intersects(area)].sort_values("key")["key"].tolist()
print(f"{len(sel)} tiles, {sel.size_mb.sum():.1f} MB")
sel[["key", "file", "year", "size_mb"]]

### 5.1 A second area: the Uedemer Hochwald
7 km west of the camp, in the forest *Uedemer Hochwald* between Marienbaum and Uedem, the topographic maps mark a field of **burial mounds** (*Hügelgräber*, K.D. = *Kulturdenkmal*). The mounds are about one metre high: invisible in a 10 m model, clear in the 1 m DGM1. These are the tiles you download in §6 and process in QGIS (geo1041, geo0741).

**Predict:** the box below is 2.8 km × 1.8 km. How many tiles? All from the same survey year?

In [ ]:
UEDEM = box(316_100, 5_729_100, 318_900, 5_730_900)      # inside the 1 km grid: touching a tile edge would count as "intersects"

sel = select_tiles(tiles, UEDEM)                          # from here on, sel = the Uedemer Hochwald
print(f"{len(sel)} tiles, {sel.size_mb.sum():.1f} MB, survey years {sorted(sel.year.unique().tolist())}")
sel[["key", "file", "year", "size_mb"]]

**Your own area:** in QGIS read the coordinates (EPSG:25832) of a place you know (*View → Panels → Coordinate Capture*, or the coordinate field in the status bar after setting the project CRS), make a `box(...)` or a `Point(...).buffer(...)` and select its tiles. Which survey year does your place have?

## 6. Download the selected tiles
Rules for downloading from a public server:
* one file after the other, not 50 at once;
* **stream** to disk in chunks, so a 4 MB file never has to fit in memory as a whole (and a 4 GB file neither);
* **skip** files you already have; compare the size with the index to notice broken downloads;
* write to a temporary name first and rename at the end, so an interrupted download never looks complete.

**Exercise:** write `download_tile(file, size_bytes, folder=ORIGINAL)` following the four rules above: return `"exists"` if the file is already there with the right size; otherwise download `DGM1_URL + file` with `stream=True` into `<file>.part` (chunks of 64 KB), check the size, rename to the final name (`Path.replace`) and return `"downloaded"`.

In [ ]:
def download_tile(file, size_bytes, folder=ORIGINAL):
    """Download one DGM1 tile into `folder` (streamed); skip it if it is already there with the right size."""
    # YOUR CODE HERE


Run it for the selected tiles of the Uedemer Hochwald (6 tiles, about 13 MB). Run the cell a second time: what changes?

In [ ]:
for row in sel.itertuples():
    t0 = time.perf_counter()
    status = download_tile(row.file, row.size_bytes)
    print(f"{row.file}  {row.size_mb:5.2f} MB  {status:10s} {time.perf_counter() - t0:5.1f} s")

### 6.1 Check one tile
Open a downloaded tile with rasterio (chapter 10, Elevation & Terrain, goes deeper).

**Predict:** are the raster bounds exactly the square of the index, or half a pixel off? What value marks "no data"?

In [ ]:
import rasterio

row = sel.iloc[0]
with rasterio.open(ORIGINAL / row.file) as src:
    dem = src.read(1, masked=True)                     # masked: nodata cells are hidden
    print(src.crs, src.res, src.shape, src.dtypes[0], "nodata", src.nodata)
    print("raster bounds", tuple(src.bounds))
    print("index square ", row.geometry.bounds)
    extent = (src.bounds.left, src.bounds.right, src.bounds.bottom, src.bounds.top)

fig, ax = plt.subplots(figsize=(7, 7))
im = ax.imshow(dem, extent=extent, cmap="terrain")
fig.colorbar(im, ax=ax, label="height (m)", shrink=0.8)
ax.set_title(row.key, fontsize=14)
ax.ticklabel_format(style="plain")
plt.show()

The raster bounds are the tile square: the corner coordinates in the name are the **outer edge** of the pixels, not the centre of the first pixel. You can mosaic the tiles without gaps or overlaps.

### 6.2 Take the tiles to your laptop
QGIS runs on your laptop, the tiles are on the hub. Pack them into **one** zip file together with the index of the selected tiles, then download it in JupyterLab (right-click on the file → *Download*) and unzip it into the `data/` folder of your course folder on the laptop.

In [ ]:
import zipfile

def zip_tiles(sel, zip_path, folder=ORIGINAL):
    """One zip with the GeoTIFFs of `sel` and a GeoPackage of their squares (layer dgm1_tiles)."""
    gpkg = zip_path.with_suffix(".gpkg")
    sel.to_file(gpkg, layer="dgm1_tiles", driver="GPKG")
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_STORED) as z:   # GeoTIFFs are compressed already
        for file in sel["file"]:
            z.write(folder / file, arcname=f"{zip_path.stem}/{file}")
        z.write(gpkg, arcname=f"{zip_path.stem}/{gpkg.name}")
    return zip_path


zip_path = zip_tiles(sel, OUT / "dgm1_uedemer_hochwald.zip")
with zipfile.ZipFile(zip_path) as z:
    print(*z.namelist(), sep="\n")
print(f"{zip_path}: {zip_path.stat().st_size / 1e6:.1f} MB")

## 7. Wrap-up
1. Look at what a server sends before you parse it: status, content type, **encoding**, size, date.
2. Download once, keep the original (`data/original/`), and write where and when it came from next to your results.
3. **BeautifulSoup** reads HTML and XML as a tree of tags with attributes; attribute values are text.
4. A **regular expression** checks the whole file name; a name that does not fit is an error, not a silent wrong number.
5. Vectorised shapely builds thousands of geometries at once; a **spatial index** finds the ones you need.
6. Download politely: one after the other, streamed, skip what you have, check the size.

**Exit ticket:** Geobasis NRW surveys your area again in 2026. Which column of your index changes, which stays the same, and what does that mean for a script that stores file names instead of keys?

**Homework:** see the course plan in the README of your course repository.